## 1. Imports & Load Raw Data

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as path

In [32]:
df = pd.read_csv('dataset/amazon_delivery.csv')
y_target = 'Delivery_Time'
df.head(5)

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys


## 2. Handle Missing / Invalid Values

In [33]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df['Order_Time'] = pd.to_timedelta(df['Order_Time'].astype(str), errors='coerce')
df['Pickup_Time'] = pd.to_timedelta(df['Pickup_Time'].astype(str), errors='coerce')

In [34]:
missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 236
Duplicates: 0
Missing Feature:
['Agent_Rating', 'Order_Time', 'Weather']


In [35]:
for col in missing_feature:
    missing_percentage = (df[col].isnull().sum() / len(df)) * 100
    
    if missing_percentage > 5.0:
        df = df.drop(columns=col)
    else:
        if pd.api.types.is_timedelta64_dtype(df[col]):
            median_timedelta = df[col].median()
            df[col] = df[col].fillna(median_timedelta)
        
        elif pd.api.types.is_datetime64_any_dtype(df[col]):
            median_date = df[col].median()
            df[col] = df[col].fillna(median_date)

        elif df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            skewness = df[col].skew()
            if abs(skewness) < 0.5:
                df[col] = df[col].fillna(df[col].mean()).round(2)
            else:
                df[col] = df[col].fillna(df[col].median()).round(2)

missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Missing Feature:\n{missing_feature}')


Missing values: 0
Missing Feature:
[]


## 3. Handling Duplicated

In [36]:
df = df.drop_duplicates(keep='last')

jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah data duplikat: {jumlah_duplikat}")

Jumlah data duplikat: 0


In [37]:
## 4. Analisis & Handling Outliers

In [38]:
feature_numerik = df.select_dtypes(include=np.number).columns.drop(y_target)
Q1 = df[feature_numerik].quantile(0.25)
Q3 = df[feature_numerik].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sebelum Dihapus: {outliers.shape[0]}")

#delete outliers
df = df.loc[((df[feature_numerik] >= lower_bound) & (df[feature_numerik] <= upper_bound)).all(axis=1)]
outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sesudah Dihapus: {outliers.shape[0]}")

Jumlah Outlier Sebelum Dihapus: 8645
Jumlah Outlier Sesudah Dihapus: 0


## 5. Feature Engineering